In [ ]:
import importlib
import itertools

import mlflow
import pandas as pd
import sklearn.ensemble

from sportsml.cbb.data.nodes import team_name_map
from sportsml.models.sklearn import SportsMLPredictor, train_sklearn
from sportsml.utils.stats import process_averages


In [ ]:
SPORT = "cbb"

features_module = importlib.import_module(f"sportsml.{SPORT}.data.features")
STATS_COLUMNS = features_module.STATS_COLUMNS
CATEGORICAL_COLUMNS = getattr(features_module, "CATEGORICAL_COLUMNS", [])
META_COLUMNS = getattr(features_module, "META_COLUMNS", [])

In [ ]:
df = pd.read_csv(f'../data/{SPORT}/data.csv')

In [ ]:
res = train_sklearn(
    games=df,
    model=sklearn.ensemble.RandomForestRegressor(n_estimators=100, verbose=4),
    stats_columns=STATS_COLUMNS,
    target_column=features_module.TARGET_COLUMN,
    season_column=features_module.SEASON_COLUMN,
    date_column=features_module.DATE_COLUMN,
    team_column=features_module.TEAM_COLUMN,
    team_opp_column=features_module.TEAM_OPP_COLUMN,
    print_metrics=True,
)

In [ ]:
model_input = pd.DataFrame(
    itertools.permutations(team_features.sort_index().index, 2), columns=["team", "opp"]
)

In [ ]:
model = mlflow.pyfunc.load_model('models/cbb')
preds = model.predict(model_input)

In [ ]:
preds = preds.assign(team=preds.team - 1101, opp=preds.opp-1101)

In [ ]:
preds = preds.pivot_table(values='preds', index='team', columns='opp')

In [ ]:
preds = preds.rename(index=team_name_map, columns=team_name_map)

In [ ]:
preds = preds.loc[
    preds.mean(axis=1).sort_values(ascending=False).index, preds.mean(axis=1).sort_values(ascending=False).index
]

In [ ]:
preds